In [ ]:
# !python -m venv .venv

In [ ]:

!pip install -U ultralytics

# !pip install -U kagglehub

# !pip install -U PyYAML Pillow pandas numpy matplotlib seaborn

# # !pip install -U torch torchvision torchaudio

# !pip install -U  scikit-learn

In [ ]:
import kagglehub
from pathlib import Path
import os
import random
import yaml
from PIL import Image
import shutil
import torch
import ultralytics
from ultralytics import YOLO
import matplotlib.pyplot as plt
import math


In [ ]:
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Download latest version
str_path = kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset")

In [6]:
path = Path(str_path)

In [7]:
trafic_data = os.path.join(path, "trafic_data")

validation_data_path =  os.path.join(trafic_data, "valid")

train_data_path =  os.path.join(trafic_data, "train")

yaml_file = os.path.join(trafic_data, "data_1.yaml")

In [8]:
def read_yaml():
    with open(yaml_file, "r", encoding="utf-8") as fichier:
        data = yaml.safe_load(fichier)
    
    return data    


yaml_data = read_yaml()

In [9]:

from pathlib import Path
from PIL import Image

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def extract_images_labels(path_name: str):
    path = Path(path_name)

    return {
        "images": sorted(
            p for p in (path / "images").rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ),
        "labels": sorted(
            p for p in (path / "labels").rglob("*.txt")
            if p.is_file()
        ),
    }


def clean_dataset(data, remove_empty_labels=False):
    # Match labels by filename stem.
    label_map = {Path(p).stem: Path(p) for p in data["labels"]}

    cleaned = {"images": [], "labels": []}

    for image in data["images"]:
        image = Path(image)
        label = label_map.get(image.stem)

        # Skip images without a corresponding label.
        if label is None:
            print("Missing label:", image.name)
            continue

        # Skip corrupted images.
        try:
            with Image.open(image) as img:
                img.verify()
        except (OSError, SyntaxError, ValueError):
            print("Corrupted image:", image)
            continue

        # Optionally remove empty label files.
        if remove_empty_labels:
            if not label.read_text(encoding="utf-8").strip():
                continue

        cleaned["images"].append(image)
        cleaned["labels"].append(label)

    return cleaned


In [10]:
validation_data = extract_images_labels(validation_data_path)
validation_data = clean_dataset(validation_data)

In [11]:

train_test_data = extract_images_labels(train_data_path)
train_test_data = clean_dataset(train_test_data)

In [12]:
data_len = len(train_test_data["images"])
test_len = int(data_len / 3)

unique_random_test_numbers = random.sample(
    range(data_len),
    test_len
)

test_indexes = set(unique_random_test_numbers)

train_data = {
    "images": [],
    "labels": []
}

test_data = {
    "images": [],
    "labels": []
}

for i in range(data_len):
    if i in test_indexes:
        test_data["images"].append(train_test_data["images"][i])
        test_data["labels"].append(train_test_data["labels"][i])
    else:
        train_data["images"].append(train_test_data["images"][i])
        train_data["labels"].append(train_test_data["labels"][i])

In [ ]:

import shutil
import yaml
from pathlib import Path

destination = Path("/content/road_data")

def copy_dataset(data, split, destination):
    images = data["images"]
    labels = data["labels"]


    for folder in ["images", "labels"]:
        (destination / split / folder).mkdir(
            parents=True, exist_ok=True
        )

    for image, label in zip(images, labels):
        image = Path(image)
        label = Path(label)

        shutil.copy2(
            image,
            destination / split / "images" / image.name
        )
        shutil.copy2(
            label,
            destination / split / "labels" / label.name
        )

copy_dataset(train_data, "train", destination)
copy_dataset(validation_data, "valid", destination)
copy_dataset(test_data, "test", destination)

# Create YOLO dataset configuration.
new_file_yaml = {}
new_file_yaml["path"] = str(destination.resolve())
new_file_yaml["train"] = "train/images"
new_file_yaml["val"] = "valid/images"
new_file_yaml["test"] = "test/images"
new_file_yaml["nc"] = yaml_data["nc"]
new_file_yaml["names"] = yaml_data["names"]

with open(destination / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(new_file_yaml, f, sort_keys=False)

print((destination / "data.yaml").read_text())


In [75]:
# %cd road_data
%ls
%pwd


data.yaml  test/  train/  valid/


'/content/road_data'

In [ ]:
model = YOLO("yolov9c.pt")

results = model.train(
    data="/content/road_data/data.yaml",
    imgsz=640,
    batch=4,
    epochs=10,
    device=0,
    project="/content/runs",
    name="road_vehicle_yolov9c",
    seed=42
)

In [ ]:
best_model = YOLO(
    "/content/runs/road_vehicle_yolov9c/weights/best.pt"
)

metrics = best_model.val(
    data="/content/road_data/data.yaml",
    split="val",
    imgsz=640,
    batch=4,
    device=0
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:

# Load your trained model
model = YOLO("/content/runs/road_vehicle_yolov9c/weights/best.pt")

test_image_p = Path("/content/road_data/test/images")

project="/content/runs"


results = model.predict(
    source= str(test_image_p),
    imgsz=640,
    conf=0.5,
    project="/content/runs",
    save=True
)

# # print(results)

# from IPython.display import display

# # Display the first prediction
# display(Image.fromarray(results[0].plot()[..., ::-1]))

In [ ]:
n = min(5, len(results))

if n == 0:
    print("Aucune image à afficher.")
else:
    cols = 2
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i in range(n):
        annotated_image = results[i].plot()

        # YOLO retourne une image en BGR : convertir en RGB
        annotated_image = annotated_image[..., ::-1]

        axes[i].imshow(annotated_image)
        axes[i].set_title(Path(results[i].path).name)
        axes[i].axis("off")

    for i in range(n, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()